In [2]:

# CELL 1: Install required packages

%pip install -q -U \
    "transformers>=4.45,<5" \
    "peft>=0.13,<1" \
    "trl>=0.12,<1" \
    "accelerate>=1.0,<2" \
    "bitsandbytes>=0.44" \
    "datasets>=3.0,<5" \
    "huggingface_hub>=0.26" \
    safetensors \
    sentencepiece \
    protobuf \
    pandas \
    pyarrow

print("All required packages installed!")

# Check PyTorch and GPU
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU RAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("WARNING: GPU is not enabled. Enable T4 GPU in Colab.")


All required packages installed!
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU RAM: 14.56 GB


In [5]:
import os
import shutil

# New workspace for LLM training
PROJECT_DIR = "/content/drive/MyDrive/PlantDisease_LLM"

# Dataset folders
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
PLANTVILLAGE_DIR = os.path.join(DATASET_DIR, "PlantVillage")

EXPERT_DIR = os.path.join(DATASET_DIR, "PlantExpertVQA")
FILTERED_QA_DIR = os.path.join(EXPERT_DIR, "filtered_QA")
KNOWLEDGE_DIR = os.path.join(EXPERT_DIR, "knowledge_cards")

# Model training folders
LLM_DIR = os.path.join(PROJECT_DIR, "LLM")
CHECKPOINT_DIR = os.path.join(LLM_DIR, "checkpoints")
LOG_DIR = os.path.join(LLM_DIR, "logs")
ADAPTER_DIR = os.path.join(LLM_DIR, "adapter")
FINAL_MODEL_DIR = os.path.join(LLM_DIR, "final_model")

# Create folders
folders = [
    PLANTVILLAGE_DIR,
    FILTERED_QA_DIR,
    KNOWLEDGE_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    ADAPTER_DIR,
    FINAL_MODEL_DIR
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Storage information
def show_storage(path, name):
    usage = shutil.disk_usage(path)
    print(f"\n{name}")
    print("Total:", round(usage.total / 1024**3, 2), "GB")
    print("Used:", round(usage.used / 1024**3, 2), "GB")
    print("Free:", round(usage.free / 1024**3, 2), "GB")

print("LLM project workspace created successfully!")
print("Project path:", PROJECT_DIR)

show_storage("/content", "Colab storage")
show_storage("/content/drive/MyDrive", "Google Drive storage")

LLM project workspace created successfully!
Project path: /content/drive/MyDrive/PlantDisease_LLM

Colab storage
Total: 112.64 GB
Used: 43.17 GB
Free: 69.45 GB

Google Drive storage
Total: 15.0 GB
Used: 0.06 GB
Free: 14.94 GB


In [6]:
import os
import json
from datasets import load_dataset, load_from_disk, ClassLabel

DATASET_ID = "geraldmc/plantvillage-full"
REVISION = "v0.1.0"

DATASET_PATH = os.path.join(PLANTVILLAGE_DIR, "plantvillage_hf")
CLASS_FILE = os.path.join(DATASET_DIR, "class_names.json")

CACHE_DIR = "/content/hf_cache_plantvillage"
os.makedirs(CACHE_DIR, exist_ok=True)

# Download only if it has not already been saved
if os.path.exists(DATASET_PATH):
    print("Loading existing PlantVillage dataset from Drive...")
    plant_dataset = load_from_disk(DATASET_PATH)
else:
    print("Downloading PlantVillage dataset...")
    plant_dataset = load_dataset(
        DATASET_ID,
        revision=REVISION,
        cache_dir=CACHE_DIR
    )

    print("Saving PlantVillage dataset to Google Drive...")
    plant_dataset.save_to_disk(DATASET_PATH)

print("\nDataset loaded successfully!")
print(plant_dataset)

# Inspect the dataset schema
split_name = next(iter(plant_dataset.keys()))
features = plant_dataset[split_name].features

print("\nDataset columns:", list(features.keys()))
print("\nDataset features:")
print(features)

# Inspect class-label metadata without assuming its format
if "class_label" in features:
    label_feature = features["class_label"]

    if isinstance(label_feature, ClassLabel):
        class_names = label_feature.names
    else:
        class_names = None
else:
    class_names = None

if class_names is not None:
    with open(CLASS_FILE, "w", encoding="utf-8") as f:
        json.dump(class_names, f, indent=4)

    print("\nClass count:", len(class_names))
    print("Class names:")
    for i, name in enumerate(class_names):
        print(i, name)

    print("\nClass mapping saved:", CLASS_FILE)
else:
    print("\nClass names are not directly encoded as ClassLabel.")
    print("We will inspect the actual label and metadata values before creating the mapping.")
    columns_to_check = [
        c for c in ["class_label", "class_idx", "host", "disease"]
        if c in features
    ]
    print("Sample records:")
    print(plant_dataset[split_name].select(range(min(5, len(plant_dataset[split_name]))))
          .select_columns(columns_to_check))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/549 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/402M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/453M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/54304 [00:00<?, ? examples/s]

Saving PlantVillage dataset to Google Drive...


Saving the dataset (0/2 shards):   0%|          | 0/54304 [00:00<?, ? examples/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['image', 'class_label', 'class_idx', 'host', 'disease', 'split', 'leaf_id', 'leaf_grouped'],
        num_rows: 54304
    })
})

Dataset columns: ['image', 'class_label', 'class_idx', 'host', 'disease', 'split', 'leaf_id', 'leaf_grouped']

Dataset features:
{'image': Image(mode=None, decode=True), 'class_label': Value('string'), 'class_idx': Value('int32'), 'host': Value('string'), 'disease': Value('string'), 'split': Value('string'), 'leaf_id': Value('string'), 'leaf_grouped': Value('bool')}

Class names are not directly encoded as ClassLabel.
We will inspect the actual label and metadata values before creating the mapping.
Sample records:
Dataset({
    features: ['class_label', 'class_idx', 'host', 'disease'],
    num_rows: 5
})


In [7]:
import pandas as pd

train_data = plant_dataset["train"]

metadata = train_data.select_columns([
    "class_label",
    "class_idx",
    "host",
    "disease"
]).to_pandas()

# Check how many unique classes exist
print("Total images:", len(metadata))
print("Unique class labels:", metadata["class_label"].nunique())
print("Unique class indices:", metadata["class_idx"].nunique())
print("Unique plant species:", metadata["host"].nunique())

# Display class mapping in numeric index order
class_mapping = (
    metadata.groupby("class_idx")
    .agg(
        class_label=("class_label", "first"),
        host=("host", "first"),
        disease=("disease", "first"),
        label_count=("class_label", "nunique")
    )
    .sort_index()
)

print("\nCOMPLETE CLASS MAPPING:")
print(class_mapping.to_string())

# Check for inconsistent mappings
print("\nConsistency checks:")
print("Indices with multiple labels:",
      (metadata.groupby("class_idx")["class_label"].nunique() > 1).sum())

print("Labels with multiple indices:",
      (metadata.groupby("class_label")["class_idx"].nunique() > 1).sum())

Total images: 54304
Unique class labels: 38
Unique class indices: 38
Unique plant species: 14

COMPLETE CLASS MAPPING:
                                                  class_label                     host                               disease  label_count
class_idx                                                                                                                                
0                                          Apple___Apple_scab                    Apple                            Apple scab            1
1                                           Apple___Black_rot                    Apple                             Black rot            1
2                                    Apple___Cedar_apple_rust                    Apple                      Cedar apple rust            1
3                                             Apple___healthy                    Apple                               healthy            1
4                                         Blueberry__

In [8]:
import json
import os

# Extract one class name for each numeric class index
class_mapping = (
    metadata[["class_idx", "class_label"]]
    .drop_duplicates()
    .sort_values("class_idx")
)

class_names = class_mapping["class_label"].tolist()

# Verify mapping
assert len(class_names) == 38
assert class_mapping["class_idx"].tolist() == list(range(38))

# Save class names
CLASS_FILE = os.path.join(DATASET_DIR, "class_names.json")

with open(CLASS_FILE, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=4, ensure_ascii=False)

print("Class mapping saved successfully!")
print("File:", CLASS_FILE)
print("Total classes:", len(class_names))

for index, name in enumerate(class_names):
    print(f"{index}: {name}")

Class mapping saved successfully!
File: /content/drive/MyDrive/PlantDisease_LLM/dataset/class_names.json
Total classes: 38
0: Apple___Apple_scab
1: Apple___Black_rot
2: Apple___Cedar_apple_rust
3: Apple___healthy
4: Blueberry___healthy
5: Cherry_(including_sour)___Powdery_mildew
6: Cherry_(including_sour)___healthy
7: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8: Corn_(maize)___Common_rust_
9: Corn_(maize)___Northern_Leaf_Blight
10: Corn_(maize)___healthy
11: Grape___Black_rot
12: Grape___Esca_(Black_Measles)
13: Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14: Grape___healthy
15: Orange___Haunglongbing_(Citrus_greening)
16: Peach___Bacterial_spot
17: Peach___healthy
18: Pepper,_bell___Bacterial_spot
19: Pepper,_bell___healthy
20: Potato___Early_blight
21: Potato___Late_blight
22: Potato___healthy
23: Raspberry___healthy
24: Soybean___healthy
25: Squash___Powdery_mildew
26: Strawberry___Leaf_scorch
27: Strawberry___healthy
28: Tomato___Bacterial_spot
29: Tomato___Early_blight
30:

In [9]:
from huggingface_hub import HfApi

REPO_ID = "SyedNazmusSakib/PlantExpertVQA"

api = HfApi()

print("Inspecting PlantExpertVQA repository...")

repo_info = api.dataset_info(
    repo_id=REPO_ID,
    files_metadata=True
)

print("\nRepository:", repo_info.id)

total_size = 0
file_details = []

for file in repo_info.siblings:
    size = getattr(file, "size", None)

    if size is not None:
        total_size += size

    file_details.append({
        "filename": file.rfilename,
        "size_mb": round(size / (1024 ** 2), 2) if size is not None else "Unknown"
    })

print("\nRepository files:")

for item in file_details:
    print(f"{item['filename']}  |  {item['size_mb']} MB")

print("\nTotal reported file size:",
      round(total_size / (1024 ** 3), 2), "GB")

print("\nNumber of repository files:", len(file_details))

Streaming output truncated to the last 5000 lines.
images/b4b00d810d5da606.jpg  |  0.09 MB
images/b4c5905bef452708.jpg  |  0.17 MB
images/b4d1ecb4355b793f.jpg  |  0.05 MB
images/b4d7b91e1a838f20.jpg  |  0.09 MB
images/b4ea5bf60abf25ea.jpg  |  0.03 MB
images/b4ecbe65d9db760b.jpg  |  0.19 MB
images/b4eedab6fca0c4f7.jpg  |  0.08 MB
images/b4f51cef1dbad401.jpg  |  0.05 MB
images/b4fdf073a9999459.jpg  |  0.04 MB
images/b512e1fcc46a9733.jpg  |  0.19 MB
images/b5192207954f018c.jpg  |  0.18 MB
images/b5205baed26311b0.jpg  |  0.34 MB
images/b54992cbb0f31471.jpg  |  0.13 MB
images/b559db575f6c99ae.jpg  |  0.32 MB
images/b5673e981174abd7.jpg  |  0.38 MB
images/b5683209c3d744dd.jpg  |  0.03 MB
images/b57ba248bde91d6e.jpg  |  0.05 MB
images/b582e1a9688c7a84.jpg  |  0.34 MB
images/b5839231e760c20d.jpg  |  0.21 MB
images/b5a24df1c7e3eead.jpg  |  0.17 MB
images/b5a49791097c828b.jpg  |  0.27 MB
images/b5a54c90107db1df.jpg  |  0.14 MB
images/b5af49aba40ecfc4.jpg  |  0.04 MB
images/b5bd8880e68d1e90.jpg  

In [10]:
# Exclude image folders and image ZIP archives
non_image_files = []

for file in repo_info.siblings:
    filename = file.rfilename
    size = getattr(file, "size", None)

    if filename.lower().startswith("images/"):
        continue

    if filename.lower().endswith((".zip", ".jpg", ".jpeg", ".png", ".webp")):
        continue

    non_image_files.append({
        "filename": filename,
        "size_mb": round(size / (1024 ** 2), 2) if size is not None else "Unknown"
    })

print("NON-IMAGE FILES:", len(non_image_files))

for item in non_image_files:
    print(f"{item['filename']} | {item['size_mb']} MB")

print("\nFile types:")
from collections import Counter

extensions = Counter(
    os.path.splitext(item["filename"])[1].lower() or "[no extension]"
    for item in non_image_files
)

for extension, count in extensions.items():
    print(extension, ":", count)

NON-IMAGE FILES: 216
.gitattributes | 0.0 MB
README.md | 0.01 MB
data/split_statistics.json | 0.0 MB
data/test.csv | 59.27 MB
data/test.jsonl | 90.72 MB
data/test.parquet | 6.24 MB
data/train.csv | 207.75 MB
data/train.jsonl | 317.96 MB
data/train.parquet | 20.73 MB
data/val.csv | 29.64 MB
data/val.jsonl | 45.35 MB
data/val.parquet | 3.18 MB
disease_kb/all_cards.jsonl | 1.37 MB
disease_kb/apple/alternaria_blotch.json | 0.01 MB
disease_kb/apple/black_rot.json | 0.01 MB
disease_kb/apple/cedar_apple_rust.json | 0.01 MB
disease_kb/apple/healthy.json | 0.01 MB
disease_kb/apple/powdery_mildew.json | 0.01 MB
disease_kb/apple/scab.json | 0.01 MB
disease_kb/arabian/jasmine_anthracnose.json | 0.01 MB
disease_kb/arabian/jasmine_arabian_jasmine_healthy.json | 0.01 MB
disease_kb/arabian/jasmine_chlorosis.json | 0.01 MB
disease_kb/arabian/jasmine_leaf_blight.json | 0.01 MB
disease_kb/arabian/jasmine_leaf_spot.json | 0.01 MB
disease_kb/arabian/jasmine_pest_damage.json | 0.01 MB
disease_kb/arabian/jas

In [11]:
import os
import json
import pandas as pd
from huggingface_hub import hf_hub_download

REPO_ID = "SyedNazmusSakib/PlantExpertVQA"

# Download only the three Parquet files and disease knowledge cards
required_files = [
    "data/train.parquet",
    "data/val.parquet",
    "data/test.parquet",
    "disease_kb/all_cards.jsonl"
]

downloaded_files = {}

for filename in required_files:
    print("Downloading:", filename)

    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=filename
    )

    downloaded_files[filename] = path
    print("Downloaded successfully.\n")

# Load Q&A splits
qa_data = {}

for split in ["train", "val", "test"]:
    filename = f"data/{split}.parquet"
    qa_data[split] = pd.read_parquet(downloaded_files[filename])

# Load disease knowledge cards
cards = []

with open(downloaded_files["disease_kb/all_cards.jsonl"], "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            cards.append(json.loads(line))

# Inspect dataset structure
print("\n========== Q&A DATASET STRUCTURE ==========")

for split, df in qa_data.items():
    print(f"\n{split.upper()}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("\nFirst record:")
    print(df.iloc[0].to_dict())

print("\n========== DISEASE KNOWLEDGE CARDS ==========")
print("Total cards:", len(cards))

if cards:
    print("\nFirst knowledge card:")
    print(json.dumps(cards[0], indent=2, ensure_ascii=False))

print("\n========== TOTAL RECORDS ==========")
print("Total Q&A:", sum(len(df) for df in qa_data.values()))
print("Total knowledge cards:", len(cards))

Downloading: data/train.parquet


data/train.parquet:   0%|          | 0.00/21.7M [00:00<?, ?B/s]

Downloaded successfully.

Downloading: data/val.parquet


data/val.parquet:   0%|          | 0.00/3.33M [00:00<?, ?B/s]

Downloaded successfully.

Downloading: data/test.parquet


data/test.parquet:   0%|          | 0.00/6.54M [00:00<?, ?B/s]

Downloaded successfully.

Downloading: disease_kb/all_cards.jsonl


all_cards.jsonl: 0.00B [00:00, ?B/s]

Downloaded successfully.


========== Q&A DATASET STRUCTURE ==========

TRAIN
Rows: 535881
Columns: ['qa_id', 'image_id', 'image_path', 'crop', 'disease', 'category', 'severity', 'question_text', 'answer', 'answer_type', 'question_category', 'cognitive_level', 'dataset_source']

First record:
{'qa_id': '7dbf480a-8364-4585-9d10-c9cec1b1d3db', 'image_id': '51517fa78a37207e.jpg', 'image_path': 'images/51517fa78a37207e.jpg', 'crop': 'bottle gourd', 'disease': 'gourd anthracnose', 'category': 'disease', 'severity': 'SEVERE', 'question_text': 'What disease is shown here?', 'answer': 'This is gourd anthracnose, displaying severe symptoms including extensive damage visible on the leaf. The symptoms begin as small, water-soaked spots and have now progressed to a severe state.', 'answer_type': '', 'question_category': 'Specific Disease Identification', 'cognitive_level': 'Level 3 (Reasoning)', 'dataset_source': 'PlantVQA'}

VAL
Rows: 76384
Columns: ['qa_id', 'image_id', 'image_path', 'crop', 'di

In [12]:
import pandas as pd
from collections import Counter

# Combine Q&A splits for inspection only
all_qa = pd.concat(
    [qa_data["train"], qa_data["val"], qa_data["test"]],
    ignore_index=True
)

print("========== Q&A CROP COVERAGE ==========")

crop_counts = (
    all_qa.groupby("crop")
    .size()
    .sort_values(ascending=False)
)

for crop, count in crop_counts.items():
    print(f"{crop}: {count:,}")

print("\n========== Q&A DISEASE COVERAGE ==========")

crop_disease_counts = (
    all_qa.groupby(["crop", "disease"])
    .size()
    .sort_values(ascending=False)
)

for (crop, disease), count in crop_disease_counts.items():
    print(f"{crop} | {disease}: {count:,}")

print("\n========== KNOWLEDGE CARD COVERAGE ==========")

card_coverage = Counter()

for card in cards:
    crop = card.get("crop", {}).get("common_name", "Unknown")
    disease = card.get("condition", {}).get("common_name", "Unknown")
    card_coverage[(crop, disease)] += 1

for (crop, disease), count in sorted(card_coverage.items()):
    print(f"{crop} | {disease}")

========== Q&A CROP COVERAGE ==========
tomato: 149,377
unknown: 95,411
corn: 79,799
mango: 75,448
orange: 52,385
grape: 39,720
rice: 38,811
litchi: 22,046
peach: 18,026
sunflower: 17,635
soybean: 16,955
potato: 15,647
apple: 14,173
peas: 13,713
cherry: 10,873
pea: 10,308
tea: 10,031
cotton: 9,813
citrus: 8,920
strawberry: 7,073
pepper: 6,033
papaya: 5,705
bottle gourd: 5,640
blueberry: 5,543
pepper bell: 4,443
squash: 4,127
rubber: 4,035
arabian jasmine: 3,481
bitter gourd: 3,364
jackfruit: 3,352
lemon: 3,316
hibiscus: 2,606
eggplant brinjal: 1,808
raspberry: 1,501
cucumber: 1,254
eggplant: 1,130
guava: 856
cauliflower: 828

========== Q&A DISEASE COVERAGE ==========
unknown | unknown: 59,358
orange | huanglongbing: 37,882
tomato | yellow leaf curl virus: 35,605
tomato | unknown: 23,526
corn | maize lethal necrosis: 20,745
corn | maize streak virus: 20,236
peach | bacterial spot: 15,947
mango | powdery mildew: 15,398
tomato | late blight: 15,395
soybean | healthy: 14,795
tomato | leaf

In [13]:
import re
import pandas as pd

def normalize(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()

# Explicit mapping: PlantVillage class -> expert dataset crop and disease names
class_aliases = {
    "Apple___Apple_scab": ("apple", ["scab", "apple scab"]),
    "Apple___Black_rot": ("apple", ["black rot"]),
    "Apple___Cedar_apple_rust": ("apple", ["cedar apple rust"]),
    "Apple___healthy": ("apple", ["healthy"]),

    "Blueberry___healthy": ("blueberry", ["healthy"]),

    "Cherry_(including_sour)___Powdery_mildew": ("cherry", ["powdery mildew"]),
    "Cherry_(including_sour)___healthy": ("cherry", ["healthy"]),

    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": ("corn", ["gray leaf spot", "cercospora leaf spot"]),
    "Corn_(maize)___Common_rust_": ("corn", ["common rust"]),
    "Corn_(maize)___Northern_Leaf_Blight": ("corn", ["northern leaf blight"]),
    "Corn_(maize)___healthy": ("corn", ["healthy"]),

    "Grape___Black_rot": ("grape", ["black rot"]),
    "Grape___Esca_(Black_Measles)": ("grape", ["esca", "esca black measles"]),
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": ("grape", ["isariopsis leaf spot"]),
    "Grape___healthy": ("grape", ["healthy"]),

    "Orange___Haunglongbing_(Citrus_greening)": ("orange", ["huanglongbing"]),

    "Peach___Bacterial_spot": ("peach", ["bacterial spot"]),
    "Peach___healthy": ("peach", ["healthy"]),

    "Pepper,_bell___Bacterial_spot": ("pepper bell", ["bacterial spot"]),
    "Pepper,_bell___healthy": ("pepper bell", ["healthy"]),

    "Potato___Early_blight": ("potato", ["early blight"]),
    "Potato___Late_blight": ("potato", ["late blight"]),
    "Potato___healthy": ("potato", ["healthy"]),

    "Raspberry___healthy": ("raspberry", ["healthy"]),
    "Soybean___healthy": ("soybean", ["healthy"]),
    "Squash___Powdery_mildew": ("squash", ["powdery mildew"]),

    "Strawberry___Leaf_scorch": ("strawberry", ["leaf scorch"]),
    "Strawberry___healthy": ("strawberry", ["healthy"]),

    "Tomato___Bacterial_spot": ("tomato", ["bacterial spot"]),
    "Tomato___Early_blight": ("tomato", ["early blight"]),
    "Tomato___Late_blight": ("tomato", ["late blight"]),
    "Tomato___Leaf_Mold": ("tomato", ["leaf mold"]),
    "Tomato___Septoria_leaf_spot": ("tomato", ["septoria leaf spot"]),
    "Tomato___Spider_mites Two-spotted_spider_mite": ("tomato", ["spider mites"]),
    "Tomato___Target_Spot": ("tomato", ["target spot"]),
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": ("tomato", ["yellow leaf curl virus"]),
    "Tomato___Tomato_mosaic_virus": ("tomato", ["mosaic virus"]),
    "Tomato___healthy": ("tomato", ["healthy"])
}

# Normalize Q&A metadata
qa_crop = all_qa["crop"].map(normalize)
qa_disease = all_qa["disease"].map(normalize)

# Check coverage for every PlantVillage class
coverage = []

for class_name in class_names:
    crop, diseases = class_aliases[class_name]

    crop_norm = normalize(crop)
    disease_norms = [normalize(d) for d in diseases]

    qa_mask = (
        (qa_crop == crop_norm) &
        (qa_disease.isin(disease_norms))
    )

    matching_cards = []

    for card in cards:
        card_crop = normalize(card.get("crop", {}).get("common_name", ""))
        condition = card.get("condition", {})

        card_diseases = [
            condition.get("common_name", "")
        ] + condition.get("alt_names", []) + card.get("aliases", [])

        card_diseases = [normalize(d) for d in card_diseases]

        if card_crop == crop_norm and any(
            d in disease_norms for d in card_diseases
        ):
            matching_cards.append(card)

    coverage.append({
        "class_index": class_names.index(class_name),
        "plantvillage_class": class_name,
        "expert_crop": crop,
        "expert_disease": " / ".join(diseases),
        "qa_records": int(qa_mask.sum()),
        "knowledge_cards": len(matching_cards)
    })

coverage_df = pd.DataFrame(coverage)

print("========== 38-CLASS EXPERT DATA COVERAGE ==========")
print(coverage_df.to_string(index=False))

print("\n========== SUMMARY ==========")
print("Classes:", len(coverage_df))
print("Classes with Q&A:", (coverage_df["qa_records"] > 0).sum())
print("Classes without Q&A:", (coverage_df["qa_records"] == 0).sum())
print("Classes with knowledge cards:", (coverage_df["knowledge_cards"] > 0).sum())
print("Classes without knowledge cards:", (coverage_df["knowledge_cards"] == 0).sum())

print("\nTotal matching Q&A records:", coverage_df["qa_records"].sum())
print("Total matching knowledge-card matches:", coverage_df["knowledge_cards"].sum())

========== 38-CLASS EXPERT DATA COVERAGE ==========
 class_index                                 plantvillage_class expert_crop                        expert_disease  qa_records  knowledge_cards
           0                                 Apple___Apple_scab       apple                     scab / apple scab        6526                1
           1                                  Apple___Black_rot       apple                             black rot        1087                1
           2                           Apple___Cedar_apple_rust       apple                      cedar apple rust         308                1
           3                                    Apple___healthy       apple                               healthy        4415                1
           4                                Blueberry___healthy   blueberry                               healthy        4352                1
           5           Cherry_(including_sour)___Powdery_mildew      cherry               

In [14]:
# Allow crop-name variations between PlantVillage and PlantExpertVQA
crop_aliases = {
    "Pepper,_bell___Bacterial_spot": ["pepper", "pepper bell"],
    "Pepper,_bell___healthy": ["pepper", "pepper bell"]
}

print("========== CORRECTED PEPPER COVERAGE ==========")

for class_name, diseases in [
    ("Pepper,_bell___Bacterial_spot", ["bacterial spot"]),
    ("Pepper,_bell___healthy", ["healthy"])
]:
    crop_options = crop_aliases[class_name]

    mask = (
        qa_crop.isin([normalize(c) for c in crop_options]) &
        qa_disease.isin([normalize(d) for d in diseases])
    )

    print(f"{class_name}: {int(mask.sum()):,} Q&A records")

print("\n========== QUESTION CATEGORY DISTRIBUTION ==========")

for class_name in class_names:
    crop, diseases = class_aliases[class_name]
    crop_options = crop_aliases.get(class_name, [crop])

    mask = (
        qa_crop.isin([normalize(c) for c in crop_options]) &
        qa_disease.isin([normalize(d) for d in diseases])
    )

    subset = all_qa.loc[mask]

    print(f"\n--- {class_name} ({len(subset):,} records) ---")

    if subset.empty:
        print("NO MATCHING RECORDS")
        continue

    categories = subset["question_category"].fillna("Unknown").value_counts()

    for category, count in categories.items():
        print(f"{category}: {count:,}")

    print("Sample questions:")
    samples = subset.drop_duplicates("question_text").head(2)

    for _, record in samples.iterrows():
        print("Q:", record["question_text"])
        print("A:", str(record["answer"])[:300])

========== CORRECTED PEPPER COVERAGE ==========
Pepper,_bell___Bacterial_spot: 4,111 Q&A records
Pepper,_bell___healthy: 4,510 Q&A records

========== QUESTION CATEGORY DISTRIBUTION ==========

--- Apple___Apple_scab (6,526 records) ---
Specific Disease Identification: 2,145
Causal Reasoning: 1,707
Counterfactual Reasoning: 1,605
Comprehensive Description: 823
: 168
Detailed Verification: 78
Sample questions:
Q: What disease is shown here?
A: This is scab, displaying severe symptoms including pale yellow or olive-green spots on the upper leaf surface that have darkened to a velvety, olive-brown, then black, often with a feathery, indistinct margin.
Q: How severe is this scab infection?
A: This is a severe scab infection. The visible pale yellow or olive-green spots that have darkened to a velvety, olive-brown, then black, with a feathery, indistinct margin, indicate advanced progression. The extensive damage shows the disease has been active for 2-3 weeks.

--- Apple___Black_rot (1,087

In [15]:
import pandas as pd

# Inspect representative classes and question categories
samples_to_check = {
    "Apple scab": ("apple", "scab"),
    "Tomato late blight": ("tomato", "late blight"),
    "Pepper bacterial spot": ("pepper bell", "bacterial spot"),
    "Potato healthy": ("potato", "healthy")
}

for title, (crop, disease) in samples_to_check.items():
    subset = all_qa[
        (all_qa["crop"].map(normalize) == normalize(crop)) &
        (all_qa["disease"].map(normalize) == normalize(disease))
    ]

    print(f"\n{'=' * 65}")
    print(f"{title} | Total records: {len(subset):,}")
    print("=" * 65)

    for category in subset["question_category"].fillna("").unique():
        category_rows = subset[
            subset["question_category"].fillna("") == category
        ]

        print(f"\nCATEGORY: {category or '[EMPTY CATEGORY]'}")
        for _, row in category_rows.sample(
            min(3, len(category_rows)), random_state=42
        ).iterrows():
            print("Q:", row["question_text"])
            print("A:", str(row["answer"])[:450])
            print("-" * 45)

# Investigate bell pepper knowledge cards
print("\n========== PEPPER KNOWLEDGE CARDS ==========")

for card in cards:
    crop_name = card.get("crop", {}).get("common_name", "")
    if "pepper" in normalize(crop_name):
        print("Crop:", crop_name)
        print("Disease:", card.get("condition", {}).get("common_name"))
        print("Aliases:", card.get("condition", {}).get("alt_names", []))
        print("-" * 45)

# Show all distinct question categories
print("\n========== ALL QUESTION CATEGORIES ==========")
print(
    all_qa["question_category"]
    .fillna("[EMPTY]")
    .value_counts()
    .to_string()
)


Apple scab | Total records: 6,526

CATEGORY: Specific Disease Identification
Q: How severe is this scab infection?
A: This is a severe scab infection. The visible extensive damage and darkening of the lesions indicate advanced progression. The moderate spread of the disease has been evident for 2-3 weeks.
---------------------------------------------
Q: What disease is shown here?
A: This is scab, displaying severe symptoms including pale yellow or olive-green spots on the upper leaf surface that have darkened to a velvety, olive-brown, and then black, often with a feathery, indistinct margin.
---------------------------------------------
Q: What disease is shown here?
A: This is scab, displaying severe symptoms including initially pale yellow or olive-green spots on the upper leaf surface. Lesions darken to a velvety, olive-brown, then black, often with a feathery, indistinct margin.
---------------------------------------------

CATEGORY: Causal Reasoning
Q: Could this severe scab h

In [16]:
import re
import pandas as pd

# Include the corrected pepper crop name
class_aliases["Pepper,_bell___Bacterial_spot"] = (
    "pepper", ["bacterial spot"]
)
class_aliases["Pepper,_bell___healthy"] = (
    "pepper", ["healthy"]
)

# Categories worth inspecting for general knowledge
allowed_categories = {
    "Causal Reasoning",
    "Comprehensive Description",
    "Counterfactual Reasoning",
    "General Health Assessment"
}

# Questions and answers that depend on an unseen image or its severity
image_dependent = re.compile(
    r"\b(this leaf|this image|this plant|this specimen|"
    r"shown here|shown in|visible here|visible damage|"
    r"what disease is shown|what disease is visible|"
    r"how severe|severity|2[-–]3 weeks|1[-–]2 weeks|"
    r"this infection|this severe|current damage|"
    r"how would this leaf|what crop is shown)\b",
    re.IGNORECASE
)

# Questions that are directly useful for class-conditioned explanations
useful_topics = re.compile(
    r"\b(cause|causes|causal|pathogen|spread|transmission|"
    r"transmitted|symptom|symptoms|management|manage|"
    r"treatment|treated|control|prevention|prevent|"
    r"environment|environmental|condition|conditions|"
    r"sanitation|fungicide|resistant|cultivar|"
    r"cultural practice|biological|chemical|"
    r"what factors|root cause|immediate action)\b",
    re.IGNORECASE
)

filtered_splits = {}

for split_name, split_df in qa_data.items():

    df = split_df.copy()

    df["crop_norm"] = df["crop"].map(normalize)
    df["disease_norm"] = df["disease"].map(normalize)

    # Match the PlantVillage crop-disease combinations
    valid_pairs = set()

    for class_name, (crop, diseases) in class_aliases.items():
        for disease in diseases:
            valid_pairs.add((normalize(crop), normalize(disease)))

    df["valid_class"] = [
        (crop, disease) in valid_pairs
        for crop, disease in zip(df["crop_norm"], df["disease_norm"])
    ]

    df["question_text"] = df["question_text"].fillna("").astype(str)
    df["answer"] = df["answer"].fillna("").astype(str)
    df["question_category"] = (
        df["question_category"].fillna("").astype(str)
    )

    # Apply quality filters
    category_ok = df["question_category"].isin(allowed_categories)

    text_ok = (
        df["question_text"].str.strip().ne("") &
        df["answer"].str.strip().ne("") &
        ~df["question_text"].str.contains(image_dependent, na=False) &
        ~df["answer"].str.contains(image_dependent, na=False) &
        df["question_text"].str.contains(useful_topics, na=False)
    )

    selected = df[df["valid_class"] & category_ok & text_ok].copy()

    filtered_splits[split_name] = selected.drop(
        columns=["crop_norm", "disease_norm", "valid_class"]
    )

    print(f"\n{split_name.upper()}")
    print("Original records:", f"{len(split_df):,}")
    print("Matching PlantVillage records:", f"{df['valid_class'].sum():,}")
    print("Retained candidate records:", f"{len(selected):,}")

    print("\nRetained category distribution:")
    print(selected["question_category"].value_counts().to_string())

    print("\nSample retained questions:")
    for _, row in selected.head(5).iterrows():
        print("Q:", row["question_text"])
        print("A:", row["answer"][:350])
        print("-" * 40)

print("\n========== TOTAL CANDIDATES ==========")
print("Train:", f"{len(filtered_splits['train']):,}")
print("Validation:", f"{len(filtered_splits['val']):,}")
print("Test:", f"{len(filtered_splits['test']):,}")

/tmp/ipykernel_3031/2472120081.py:76: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["question_text"].str.contains(image_dependent, na=False) &
/tmp/ipykernel_3031/2472120081.py:77: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["answer"].str.contains(image_dependent, na=False) &
/tmp/ipykernel_3031/2472120081.py:78: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["question_text"].str.contains(useful_topics, na=False)



TRAIN
Original records: 535,881
Matching PlantVillage records: 205,959
Retained candidate records: 13,711

Retained category distribution:
question_category
Causal Reasoning             13324
Counterfactual Reasoning       351
Comprehensive Description       36

Sample retained questions:
Q: What conditions led to this early blight infection?
A: early blight developed due to high humidity and warm temperatures. The moderate progression visible suggests these conditions have been favorable for fungal growth. These factors have created a moist environment conducive to fungal spore germination and infection.
----------------------------------------
Q: What conditions led to this early blight infection?
A: early blight developed due to high humidity and poor air circulation. The moderate progression visible suggests these conditions have been favorable for fungal growth. These factors have created a conducive environment for the fungus to thrive and spread.
-------------------------------

/tmp/ipykernel_3031/2472120081.py:76: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["question_text"].str.contains(image_dependent, na=False) &
/tmp/ipykernel_3031/2472120081.py:77: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["answer"].str.contains(image_dependent, na=False) &
/tmp/ipykernel_3031/2472120081.py:78: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["question_text"].str.contains(useful_topics, na=False)



VAL
Original records: 76,384
Matching PlantVillage records: 29,245
Retained candidate records: 1,970

Retained category distribution:
question_category
Causal Reasoning             1917
Counterfactual Reasoning       51
Comprehensive Description       2

Sample retained questions:
Q: What conditions led to this early blight infection?
A: early blight developed due to high humidity and poor air circulation. The moderate progression visible suggests favorable conditions for fungal growth. These factors have created a conducive environment for the pathogen to thrive and spread.
----------------------------------------
Q: What conditions led to this early blight infection?
A: early blight developed due to high humidity and poor air circulation. The moderate progression visible suggests these conditions have been present for a while. These factors allowed the fungus to thrive and spread, leading to the visible symptoms.
----------------------------------------
Q: What conditions led to thi

/tmp/ipykernel_3031/2472120081.py:76: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["question_text"].str.contains(image_dependent, na=False) &
/tmp/ipykernel_3031/2472120081.py:77: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["answer"].str.contains(image_dependent, na=False) &



TEST
Original records: 152,921
Matching PlantVillage records: 58,399
Retained candidate records: 3,893

Retained category distribution:
question_category
Causal Reasoning             3778
Counterfactual Reasoning      104
Comprehensive Description      11

Sample retained questions:
Q: What conditions led to this early blight infection?
A: early blight developed due to high humidity and moist conditions. The moderate progression visible suggests optimal conditions for fungal spore germination and infection. These factors provide a conducive environment for the fungus to thrive and spread.
----------------------------------------
Q: What conditions led to this early blight infection?
A: early blight developed due to high humidity and poor air circulation. The moderate progression visible suggests these conditions have been present for a few weeks. These factors favor the growth and spread of the fungal pathogen, leading to the current infection stage.
----------------------------------

/tmp/ipykernel_3031/2472120081.py:78: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["question_text"].str.contains(useful_topics, na=False)


In [17]:
import pandas as pd

for split_name, df in filtered_splits.items():

    print(f"\n{'=' * 65}")
    print(f"{split_name.upper()} CLASS DISTRIBUTION")
    print("=" * 65)

    counts = (
        df.groupby(["crop", "disease"])
        .size()
        .sort_values(ascending=False)
    )

    print(counts.to_string())

    print("\nUnique crop-disease combinations:", len(counts))
    print("Total records:", len(df))

print("\n========== TRAINING CATEGORY AND CLASS SUMMARY ==========")

train_summary = (
    filtered_splits["train"]
    .groupby(["crop", "disease", "question_category"])
    .size()
    .unstack(fill_value=0)
)

print(train_summary.to_string())


TRAIN CLASS DISTRIBUTION
crop        disease               
orange      huanglongbing             3085
tomato      yellow leaf curl virus    2701
peach       bacterial spot            1410
tomato      target spot                980
            early blight               972
potato      late blight                915
grape       esca (black measles)       646
cherry      powdery mildew             568
tomato      bacterial spot             506
            late blight                418
            septoria leaf spot         402
potato      early blight               331
tomato      mosaic virus               268
            spider mites               152
grape       healthy                     65
pepper      bacterial spot              65
corn        healthy                     53
strawberry  healthy                     32
apple       healthy                     28
tomato      healthy                     26
apple       scab                        21
cherry      healthy                 

In [18]:
import pandas as pd

def card_matches_class(card, crop, disease_aliases):
    card_crop = normalize(card.get("crop", {}).get("common_name", ""))

    crop_aliases = {
        "pepper": ["pepper", "bell pepper", "pepper bell"],
        "corn": ["corn", "maize"],
        "cherry": ["cherry", "sour cherry"],
    }

    valid_crops = [normalize(x) for x in crop_aliases.get(crop, [crop])]

    if card_crop not in valid_crops:
        return False

    condition = card.get("condition", {})

    card_diseases = [
        condition.get("common_name", ""),
        *condition.get("alt_names", []),
        *card.get("aliases", [])
    ]

    card_diseases = [normalize(x) for x in card_diseases]

    expected = [normalize(x) for x in disease_aliases]

    return any(
        a == b or a in b or b in a
        for a in expected
        for b in card_diseases
        if a and b
    )


card_report = []

for class_name in class_names:
    crop, diseases = class_aliases[class_name]

    matches = [
        card for card in cards
        if card_matches_class(card, crop, diseases)
    ]

    card_report.append({
        "class": class_name,
        "matching_cards": len(matches),
        "card_disease_names": [
            card.get("condition", {}).get("common_name", "")
            for card in matches
        ]
    })

card_report_df = pd.DataFrame(card_report)

print("========== KNOWLEDGE CARD COVERAGE ==========")
print(card_report_df.to_string(index=False))

print("\nClasses covered:", (card_report_df["matching_cards"] > 0).sum())
print("Classes missing cards:", (card_report_df["matching_cards"] == 0).sum())

print("\n========== CARD CONTENT STRUCTURE ==========")

for card in cards:
    if card_matches_class(card, "apple", ["apple scab", "scab"]):
        print("Disease:", card.get("condition", {}).get("common_name"))
        print("Pathogen:", card.get("condition", {}).get("pathogen"))
        print("Symptoms sections:", list(card.get("symptoms", {}).keys()))
        print("Management sections:", list(card.get("management", {}).keys()))
        print("Transmission:", card.get("transmission"))
        print("Environmental risk:", card.get("environmental_risk"))
        break

========== KNOWLEDGE CARD COVERAGE ==========
                                             class  matching_cards                        card_disease_names
                                Apple___Apple_scab               1                                    [scab]
                                 Apple___Black_rot               1                               [black rot]
                          Apple___Cedar_apple_rust               1                        [cedar apple rust]
                                   Apple___healthy               1                                 [healthy]
                               Blueberry___healthy               1                                 [healthy]
          Cherry_(including_sour)___Powdery_mildew               1                          [powdery mildew]
                 Cherry_(including_sour)___healthy               1                                 [healthy]
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot               1                

In [19]:
print("========== AMBIGUOUS KNOWLEDGE CARD MATCHES ==========")

for _, row in card_report_df.iterrows():

    if row["matching_cards"] > 1:

        class_name = row["class"]
        crop, diseases = class_aliases[class_name]

        matches = [
            card for card in cards
            if card_matches_class(card, crop, diseases)
        ]

        print(f"\nCLASS: {class_name}")

        for i, card in enumerate(matches, 1):
            print(f"\nCard {i}")
            print("Disease ID:", card.get("disease_id"))
            print("Crop:", card.get("crop", {}).get("common_name"))
            print(
                "Condition:",
                card.get("condition", {}).get("common_name")
            )
            print(
                "Alternative names:",
                card.get("condition", {}).get("alt_names")
            )
            print("Aliases:", card.get("aliases"))
            print("-" * 50)

========== AMBIGUOUS KNOWLEDGE CARD MATCHES ==========

CLASS: Tomato___Target_Spot

Card 1
Disease ID: tomato.disease_fungal.early_blight
Crop: tomato
Condition: early blight
Alternative names: ['Alternaria tomatophila']
Aliases: ['Alternaria leaf spot', 'target spot of tomato']
--------------------------------------------------

Card 2
Disease ID: tomato.disease_fungal.target_spot
Crop: Tomato
Condition: Target Spot
Alternative names: ['Corynespora leaf spot']
Aliases: ['Corynespora leaf spot']
--------------------------------------------------

CLASS: Tomato___Tomato_Yellow_Leaf_Curl_Virus

Card 1
Disease ID: tomato.disease_viral.leaf_curl_virus
Crop: tomato
Condition: Leaf Curl Virus
Alternative names: ['TLCV', 'Tomato leaf curl']
Aliases: ['TLCV', 'Tomato leaf curl disease']
--------------------------------------------------

Card 2
Disease ID: tomato.disease_viral.yellow_leaf_curl_virus
Crop: tomato
Condition: yellow leaf curl virus
Alternative names: ['TYLCV']
Aliases: ['TYLCV',

In [20]:
# Resolve the two ambiguous classes and retain the unique matches.
preferred_cards = {
    "Tomato___Target_Spot": "tomato.disease_fungal.target_spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus":
        "tomato.disease_viral.yellow_leaf_curl_virus"
}

class_knowledge = {}

for class_name in class_names:
    crop, diseases = class_aliases[class_name]

    matches = [
        card for card in cards
        if card_matches_class(card, crop, diseases)
    ]

    preferred_id = preferred_cards.get(class_name)

    if preferred_id:
        matches = [
            card for card in matches
            if card.get("disease_id") == preferred_id
        ]

    if len(matches) != 1:
        raise ValueError(
            f"{class_name}: expected exactly 1 card, found {len(matches)}"
        )

    class_knowledge[class_name] = matches[0]

print("Knowledge cards mapped:", len(class_knowledge))
print("PlantVillage classes:", len(class_names))
print("All classes mapped:", set(class_knowledge) == set(class_names))

Knowledge cards mapped: 38
PlantVillage classes: 38
All classes mapped: True


In [22]:
from pathlib import Path

VQA_DIR = Path(
    "/content/drive/MyDrive/PlantDisease_LLM/dataset/PlantExpertVQA"
)

VQA_DIR.mkdir(parents=True, exist_ok=True)

print("VQA_DIR:", VQA_DIR)
print("Folder exists:", VQA_DIR.exists())

VQA_DIR: /content/drive/MyDrive/PlantDisease_LLM/dataset/PlantExpertVQA
Folder exists: True


In [25]:
import json
import re
import random
from collections import defaultdict

random.seed(42)

image_pattern = re.compile(
    r"\b(visible|visibly|shown here|shown in|this leaf|"
    r"this image|this specimen|moderate progression|"
    r"current damage|progression visible|this severe|"
    r"this infection|2[-–]3 weeks|1[-–]2 weeks)\b",
    re.IGNORECASE
)

class_lookup = {
    (normalize(crop), normalize(disease)): name
    for name, (crop, diseases) in class_aliases.items()
    for disease in diseases
}

system_prompt = (
    "You are a plant disease knowledge assistant. "
    "Use reliable disease knowledge. Never claim to inspect an image. "
    "Do not invent facts or treatments. State uncertainty when needed."
)

final_splits = {}

for split_name, df in filtered_splits.items():

    grouped = defaultdict(list)

    for _, row in df.iterrows():

        class_name = class_lookup.get((
            normalize(row["crop"]),
            normalize(row["disease"])
        ))

        question = str(row["question_text"]).strip()
        answer = str(row["answer"]).strip()

        if not class_name or not question or not answer:
            continue

        if image_pattern.search(question) or image_pattern.search(answer):
            continue

        if row["question_category"] not in {
            "Causal Reasoning",
            "Comprehensive Description"
        }:
            continue

        grouped[class_name].append({
            "class_name": class_name,
            "question": question,
            "answer": answer
        })

    selected = []

    limit = 250 if split_name == "train" else 50

    for class_name, examples in grouped.items():
        unique = {
            (x["question"], x["answer"]): x
            for x in examples
        }

        examples = list(unique.values())
        random.shuffle(examples)
        selected.extend(examples[:limit])

    final_splits[split_name] = selected

    output_path = VQA_DIR / f"{split_name}_sft.jsonl"

    with open(output_path, "w", encoding="utf-8") as f:
        for item in selected:
            example = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": (
                            f"Plant disease class: {item['class_name']}\n"
                            f"Question: {item['question']}"
                        )
                    },
                    {"role": "assistant", "content": item["answer"]}
                ]
            }
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

    print(f"{split_name}: {len(selected):,} examples saved")

with open(
    VQA_DIR / "class_knowledge_cards.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(class_knowledge, f, indent=2, ensure_ascii=False)

print("\n38 knowledge cards saved.")
print("Fine-tuning data preparation completed.")

train: 243 examples saved
val: 147 examples saved
test: 175 examples saved

38 knowledge cards saved.
Fine-tuning data preparation completed.


In [27]:
from huggingface_hub import login

login()

In [29]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"

print("Loading Qwen3-4B tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Qwen3-4B tokenizer loaded successfully!")

Loading Qwen3-4B tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Qwen3-4B tokenizer loaded successfully!


In [30]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Loading Qwen3-4B in 4-bit...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Qwen3-4B loaded successfully!")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading Qwen3-4B in 4-bit...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-4B loaded successfully!
GPU memory allocated: 2.48 GB


In [31]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

print("Preparing Qwen3-4B for QLoRA...")

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("QLoRA adapter attached successfully!")

Preparing Qwen3-4B for QLoRA...
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
QLoRA adapter attached successfully!


In [32]:
import json
from datasets import Dataset

def load_sft(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            text = tokenizer.apply_chat_template(
                item["messages"],
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False
            )
            rows.append({"text": text})
    return Dataset.from_list(rows)

train_dataset = load_sft(VQA_DIR / "train_sft.jsonl")
val_dataset = load_sft(VQA_DIR / "val_sft.jsonl")

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Training examples: 243
Validation examples: 147


In [33]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

print("Tokenizing training and validation data...")

def tokenize_dataset(dataset):
    return dataset.map(
        lambda x: tokenizer(
            x["text"],
            truncation=True,
            max_length=1024
        ),
        batched=True,
        remove_columns=["text"]
    )

tokenized_train = tokenize_dataset(train_dataset)
tokenized_val = tokenize_dataset(val_dataset)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=5,
    load_best_model_at_end=True,
    report_to="none",
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    ),
    processing_class=tokenizer
)

print("Starting Qwen3-4B QLoRA fine-tuning...")

trainer.train()

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Fine-tuning completed!")
print("Adapter saved to:", ADAPTER_DIR)

Tokenizing training and validation data...


Map:   0%|          | 0/243 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting Qwen3-4B QLoRA fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.209500,0.134705
2,0.134400,0.092098
3,0.101300,0.080861


Fine-tuning completed!
Adapter saved to: /content/drive/MyDrive/PlantDisease_LLM/LLM/adapter


In [34]:
model.eval()
model.config.use_cache = True

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

messages = [
    {
        "role": "system",
        "content": (
            "You are a plant disease expert assistant. "
            "Explain disease causes, symptoms, spread, management "
            "and prevention accurately. Do not claim to inspect images."
        )
    },
    {
        "role": "user",
        "content": (
            "Plant disease class: Tomato___Early_blight\n"
            "Question: What causes early blight in tomato plants? "
            "Explain its symptoms, spread, treatment and prevention."
        )
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=False
)

inputs = inputs.to(model.get_input_embeddings().weight.device)

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens=True
)

print("QWEN3-4B ANSWER:\n")
print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


QWEN3-4B ANSWER:

Early blight develops due to the fungal pathogen Alternaria solani. These lesions are characterized by the 'target-like' appearance, with concentric rings. The plant is in the early blight stage, marked by these lesions. The fungus spreads via spores, often through splashing water. Conditions are humid and warm, which favor the development of these lesions. Treatment involves using a fungicide like chlorothalonil or copper. Apply these treatments every 7-10 days. Also, ensure good air circulation and remove infected leaves to prevent further spread. Maintain a dry environment to inhibit the fungus. These measures will help control the infection and prevent the development of more lesions. Monitor the plant regularly for any new signs of infection.


In [35]:
import json
import torch

class_name = "Tomato___Early_blight"
card = class_knowledge[class_name]

messages = [
    {
        "role": "system",
        "content": (
            "You are a plant disease knowledge assistant. "
            "Use the supplied knowledge card as your factual source. "
            "Never claim to inspect an image. "
            "Do not invent treatments, dosages or application schedules. "
            "Mention uncertainty when information is unavailable."
        )
    },
    {
        "role": "user",
        "content": (
            f"Plant disease class: {class_name}\n\n"
            f"Knowledge card:\n{json.dumps(card, ensure_ascii=False)}\n\n"
            "Explain the cause, symptoms, spread, management and prevention."
        )
    }
]

encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=False,
    return_dict=True
)

encoded = {
    key: value.to(model.get_input_embeddings().weight.device)
    for key, value in encoded.items()
}

with torch.no_grad():
    output = model.generate(
        **encoded,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    output[0][encoded["input_ids"].shape[-1]:],
    skip_special_tokens=True
))

This is Early Blight, identifiable by the 'target-like' lesions with concentric rings. Note the dark brown to black spots and the distinct yellow halos around them. These symptoms are characteristic of Alternaria solani, the causal agent. The plant is showing signs of progressive defoliation, which is a concern. Management involves using fungicides like chlorothalonil or mancozeb, along with cultural practices such as crop rotation and proper spacing. Also, remove infected leaves to reduce the spread of the disease. Monitor weather conditions to apply treatments when humidity is high, as this favors the development of the pathogen.


In [37]:
import os

print("Adapter folder:", ADAPTER_DIR)

for file in sorted(os.listdir(ADAPTER_DIR)):
    path = os.path.join(ADAPTER_DIR, file)
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024**2)
        print(f"{file}: {size:.2f} MB")

Adapter folder: /content/drive/MyDrive/PlantDisease_LLM/LLM/adapter
README.md: 0.00 MB
adapter_config.json: 0.00 MB
adapter_model.safetensors: 126.06 MB
added_tokens.json: 0.00 MB
chat_template.jinja: 0.00 MB
merges.txt: 1.59 MB
special_tokens_map.json: 0.00 MB
tokenizer.json: 10.89 MB
tokenizer_config.json: 0.01 MB
vocab.json: 2.65 MB


In [39]:
import psutil

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total / 1024**3:.2f} GB")
print(f"Available RAM: {ram.available / 1024**3:.2f} GB")

Total RAM: 12.67 GB
Available RAM: 5.95 GB


In [40]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("Reloading Qwen3-4B in 4-bit...")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Attaching trained adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model.eval()
model.config.use_cache = True

print("Fine-tuned Qwen3-4B loaded successfully!")
print("GPU memory:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")

Reloading Qwen3-4B in 4-bit...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Attaching trained adapter...
Fine-tuned Qwen3-4B loaded successfully!
GPU memory: 4.07 GB


In [41]:
from huggingface_hub import hf_hub_download
import os

MODEL_SAVE_DIR = "/content/drive/MyDrive/PlantDisease_LLM/LLM/base_model"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

model_path = hf_hub_download(
    repo_id="Qwen/Qwen3-4B-GGUF",
    filename="Qwen3-4B-Q4_K_M.gguf",
    local_dir=MODEL_SAVE_DIR
)

print("Base model downloaded successfully!")
print(model_path)
print("Size:", round(os.path.getsize(model_path)/1024**3, 2), "GB")

Qwen3-4B-Q4_K_M.gguf:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

Base model downloaded successfully!
/content/drive/MyDrive/PlantDisease_LLM/LLM/base_model/Qwen3-4B-Q4_K_M.gguf
Size: 2.33 GB


In [42]:
import os
import subprocess

ADAPTER_DIR = "/content/drive/MyDrive/PlantDisease_LLM/LLM/adapter"
OUTPUT_DIR = "/content/drive/MyDrive/PlantDisease_LLM/LLM"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "plant_disease_adapter.gguf")
LLAMA_CPP_DIR = "/content/llama.cpp"

# Verify the original adapter
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json"))
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))

# Download the official converter repository
if not os.path.exists(LLAMA_CPP_DIR):
    subprocess.run([
        "git", "clone", "--depth", "1", "--recurse-submodules",
        "https://github.com/ggml-org/llama.cpp.git",
        LLAMA_CPP_DIR
    ], check=True)

# Convert the trained PEFT adapter to GGUF
subprocess.run([
    "python",
    os.path.join(LLAMA_CPP_DIR, "convert_lora_to_gguf.py"),
    "--base-model-id", "Qwen/Qwen3-4B",
    "--outfile", OUTPUT_FILE,
    "--outtype", "f16",
    ADAPTER_DIR
], check=True)

print("\nAdapter conversion completed!")
print("File:", OUTPUT_FILE)
print("Size:", round(os.path.getsize(OUTPUT_FILE) / (1024**2), 2), "MB")


Adapter conversion completed!
File: /content/drive/MyDrive/PlantDisease_LLM/LLM/plant_disease_adapter.gguf
Size: 63.03 MB
